# FloodNet: Street Flooding in New York City

## Reproducible descriptive analysis

This notebook uses the saved cleaned event extract and sensor-priority summary to reproduce descriptive counts, deployment-normalized screening rates, and a threshold-sensitivity check.

**Snapshot limitation:** the saved project data are from August 2026 and may not match the current public feed.

**Project-defined screening rules (not official FloodNet definitions):**
- Severe: maximum depth ≥ 4 inches
- Persistent: total event duration ≥ 120 minutes
- Severe-and-persistent: both conditions in the same event

Expected files: `data/FloodNet_clean_event_dataset.csv` and `data/FloodNet_sensor_priority_summary.csv`.

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

DATA_DIR = Path("data")
events = pd.read_csv(DATA_DIR / "FloodNet_clean_event_dataset.csv", low_memory=False)
priority = pd.read_csv(DATA_DIR / "FloodNet_sensor_priority_summary.csv", low_memory=False)

print(f"Event rows: {len(events):,}")
print(f"Unique sensors: {events['Sensor ID'].nunique():,}")
print(f"Priority summary rows: {len(priority):,}")

## 1. Validate fields and define the outcome

In [ ]:
required = ["Sensor ID", "Maximum Flood Depth (inches)", "Total Duration (minutes)", "valid_deployment", "severe_persistent"]
missing = [c for c in required if c not in events.columns]
if missing:
    raise ValueError(f"Missing required event columns: {missing}")

for col in ["Maximum Flood Depth (inches)", "Total Duration (minutes)"]:
    events[col] = pd.to_numeric(events[col], errors="coerce")

def as_bool(s):
    if s.dtype == bool:
        return s
    return s.astype(str).str.strip().str.lower().map({"true": True, "false": False, "1": True, "0": False, "yes": True, "no": False})

events["valid_deployment"] = as_bool(events["valid_deployment"])
events["saved_outcome"] = as_bool(events["severe_persistent"])
analysis = events.loc[events["valid_deployment"].fillna(False)].copy()

DEPTH_CUTOFF = 4.0
DURATION_CUTOFF = 120.0
analysis["severe_recalc"] = analysis["Maximum Flood Depth (inches)"] >= DEPTH_CUTOFF
analysis["persistent_recalc"] = analysis["Total Duration (minutes)"] >= DURATION_CUTOFF
analysis["severe_persistent_recalc"] = analysis["severe_recalc"] & analysis["persistent_recalc"]

pd.DataFrame({
    "Measure": ["Analysis events", "Severe events", "Persistent events", "Severe AND persistent events"],
    "Count": [len(analysis), int(analysis.severe_recalc.sum()), int(analysis.persistent_recalc.sum()), int(analysis.severe_persistent_recalc.sum())]
})

## 2. Compare saved and recalculated outcome flags

In [ ]:
pd.crosstab(
    analysis["saved_outcome"].fillna(False),
    analysis["severe_persistent_recalc"],
    rownames=["Saved outcome flag"],
    colnames=["Recalculated outcome flag"],
    dropna=False
)

## 3. Deployment-normalized screening rates

In [ ]:
required = ["Sensor ID", "events", "exposure_days", "severe_persistent_events", "Sensor Name", "Borough"]
missing = [c for c in required if c not in priority.columns]
if missing:
    raise ValueError(f"Missing required priority columns: {missing}")

MIN_EVENTS = 30
ranked = priority.loc[priority["events"] >= MIN_EVENTS].copy()
ranked["recalculated_rate"] = ranked["severe_persistent_events"] / ranked["exposure_days"] * 100
ranked = ranked.sort_values("recalculated_rate", ascending=False)

ranked[["Sensor ID", "Sensor Name", "Borough", "events", "exposure_days", "severe_persistent_events", "recalculated_rate"]].head(10).round({"recalculated_rate": 2})

In [ ]:
top = ranked.head(10).sort_values("recalculated_rate")
plt.figure(figsize=(10, 6))
plt.barh(top["Sensor Name"].astype(str), top["recalculated_rate"])
plt.xlabel("Severe-and-persistent events per 100 deployment-days")
plt.ylabel("Sensor location")
plt.title("Highest normalized screening rates (minimum 30 events)")
plt.tight_layout()
plt.show()

## 4. Threshold sensitivity

In [ ]:
rows = []
for depth_cutoff in [4, 8, 12]:
    for duration_cutoff in [60, 120, 240]:
        qualifies = ((analysis["Maximum Flood Depth (inches)"] >= depth_cutoff) &
                     (analysis["Total Duration (minutes)"] >= duration_cutoff))
        rows.append({"Depth threshold (inches)": depth_cutoff,
                     "Duration threshold (minutes)": duration_cutoff,
                     "Qualifying events": int(qualifies.sum())})
sensitivity = pd.DataFrame(rows)
sensitivity.pivot(index="Depth threshold (inches)", columns="Duration threshold (minutes)", values="Qualifying events")

## 5. Interpretation and limitations

- Sensors are intentionally placed and are not a random sample of every NYC street.
- Deployment days are a nominal exposure measure and may not equal actual sensor uptime.
- Thresholds and the minimum-event filter are project choices; rankings may change under alternatives.
- Repeated events from one sensor are not independent locations.
- These results do not directly measure damages, injuries, property loss, or neighborhood-wide risk.
- This notebook starts from a cleaned snapshot; it does not reproduce every step from raw public downloads.
- The project's adjusted GEE coefficients are not included as final claims because outcome coding and model specification need a verified rerun.

**Defensible conclusion:** treat high normalized rates as candidates for closer review, validate the locations and data quality, and check whether the shortlist persists under alternative thresholds. Do not call this a definitive ranking of NYC streets.